# Silent Signal — ASL Citizen top 200 và RTMPose trên Colab

Notebook **độc lập** này tự chuẩn bị mọi thành phần cần thiết: lấy ASL Citizen khi runtime chưa có dữ liệu, tạo manifest chính thức, cài môi trường RTMPose cố định, chọn **200 lớp ASL Citizen phổ biến nhất theo điểm tần suất hội thoại của ASL-LEX 2.0**, rồi trích xuất 133 keypoint COCO-WholeBody và lưu lên Drive. Không cần chạy notebook `00` hay `03` trước.

Trước khi chạy:

1. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100).
2. Đọc và chấp nhận điều khoản ASL Citizen trước khi bật tải dataset.
3. Nhánh trong `PROJECT_GIT_REF` phải tồn tại trên GitHub. Đổi thành `dev` sau khi merge.

Mặc định notebook chạy pilot trước rồi chạy full extraction. Pose cache, model, manifest và báo cáo đều nằm trên Drive; dataset giải nén và môi trường Python nằm trong `/content` nên phải tạo lại sau khi Colab reset.


## Tiêu chí và giấy phép

`SignFrequency(M)` trong ASL-LEX 2.0 là điểm trung bình 1–7 về mức độ thường gặp của dấu hiệu trong hội thoại ASL hằng ngày. Đây là tiêu chí xếp hạng; **không dùng số video của ASL Citizen làm tần suất từ**. Khi bằng điểm, notebook chỉ dùng tên gloss và class index gốc để tạo thứ tự tất định.

Nguồn: [ASL-LEX downloads](https://asl-lex.org/download.html), [bài báo ASL-LEX 2.0](https://doi.org/10.1093/deafed/enaa038). Database/visualization được công bố theo CC BY-NC 4.0; kiểm tra [điều khoản hiện tại](https://asl-lex.org/about.html) và trích dẫn bài báo khi sử dụng. Notebook chỉ tải CSV biến từ vựng, không tải hoặc tái sử dụng video dấu hiệu của ASL-LEX.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-top200-pose-extraction'  # @param {type:'string'}
PROJECT_ROOT = Path('/content/silent-signal')
DATASET_ROOT = Path('/content/ASL_Citizen')  # @param {type:'string'}
STREAM_EXTRACT_DATASET = True  # @param {type:'boolean'}
ACCEPT_ASL_CITIZEN_LICENSE = False  # @param {type:'boolean'}
PREPARE_MANIFEST = True  # @param {type:'boolean'}
PREPARATION_ROOT = Path('/content/drive/MyDrive/silent-signal-results/asl_citizen')
SOURCE_MANIFEST = PREPARATION_ROOT / 'manifests/asl_citizen.csv'

ASL_LEX_URL = 'https://osf.io/download/9nygd'
ASL_LEX_ROOT = PREPARATION_ROOT / 'references/asl_lex_2_0'
ASL_LEX_CSV = ASL_LEX_ROOT / 'signdata.csv'
CLASS_COUNT = 200  # @param {type:'integer'}
SUBSETS_ROOT = PREPARATION_ROOT / 'subsets'
SUBSET_NAME = f'asl_citizen_asllex_top{CLASS_COUNT}'
SUBSET_ROOT = SUBSETS_ROOT / SUBSET_NAME
SUBSET_MANIFEST = SUBSET_ROOT / 'manifest.csv'
SELECTION_REPORT = SUBSET_ROOT / 'selection_report.json'

POSE_ENV_ROOT = Path('/content/pose-env')
MMPOSE_ROOT = Path('/content/mmpose-v1.3.2')
MODEL_ROOT = Path('/content/drive/MyDrive/silent-signal-models/openmmlab')
DOWNLOAD_MODELS = True  # @param {type:'boolean'}
POSE_CHECKPOINT = MODEL_ROOT / 'rtmpose-l-wholebody-384x288.pth'
DET_CHECKPOINT = MODEL_ROOT / 'rtmdet-m-person.pth'
PINNED_CONFIG = (
    PREPARATION_ROOT
    / 'pose/rtmpose_l_coco_wholebody_384x288/provenance/rtmpose-colab-pinned.yaml'
)
POSE_ROOT = SUBSET_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288'
POSE_OUTPUT_ROOT = POSE_ROOT / 'raw'

RUN_SELECTION = True  # @param {type:'boolean'}
RUN_PILOT = True  # @param {type:'boolean'}
PILOT_LIMIT = 20  # @param {type:'integer'}
RUN_FULL_EXTRACTION = True  # @param {type:'boolean'}
NUM_SHARDS = 1  # @param {type:'integer'}
SHARD_INDEX = 0  # @param {type:'integer'}
OVERWRITE = False  # @param {type:'boolean'}

if CLASS_COUNT < 1 or PILOT_LIMIT < 1:
    raise ValueError('CLASS_COUNT và PILOT_LIMIT phải lớn hơn 0.')
if NUM_SHARDS < 1 or not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError('Cần 0 <= SHARD_INDEX < NUM_SHARDS.')
ASL_LEX_ROOT.mkdir(parents=True, exist_ok=True)
SUBSETS_ROOT.mkdir(parents=True, exist_ok=True)
POSE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print('Source manifest:', SOURCE_MANIFEST)
print('Subset output:', SUBSET_ROOT)
print('Pose cache:', POSE_OUTPUT_ROOT)


## Lấy đúng phiên bản mã nguồn

Cell này không thay đổi repository trên máy cá nhân. Nó chỉ clone/cập nhật bản làm việc tạm trong Colab. Nếu nhánh feature chưa được push thì cell sẽ dừng; không tự chuyển sang mã cũ trên `dev`.


In [ ]:
import subprocess

def run(command, **kwargs):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, check=True, **kwargs)

def git_output(root, *arguments):
    return subprocess.check_output(
        ['git', '-C', str(root), *arguments], text=True
    ).strip()

if not PROJECT_ROOT.exists():
    run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--depth', '1',
        PROJECT_GIT_URL, str(PROJECT_ROOT),
    ])
else:
    if not (PROJECT_ROOT / '.git').is_dir():
        raise RuntimeError('PROJECT_ROOT tồn tại nhưng không phải Git repository.')
    if git_output(PROJECT_ROOT, 'remote', 'get-url', 'origin') != PROJECT_GIT_URL:
        raise RuntimeError('Repository Colab có origin khác PROJECT_GIT_URL.')
    if git_output(PROJECT_ROOT, 'status', '--porcelain'):
        raise RuntimeError('Repository Colab có thay đổi chưa commit; không tự ghi đè.')
    run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', PROJECT_GIT_REF])
    local_ref = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'show-ref', '--verify', '--quiet',
         f'refs/heads/{PROJECT_GIT_REF}']
    ).returncode == 0
    if local_ref:
        run(['git', '-C', str(PROJECT_ROOT), 'checkout', PROJECT_GIT_REF])
        run(['git', '-C', str(PROJECT_ROOT), 'merge', '--ff-only', 'FETCH_HEAD'])
    else:
        run([
            'git', '-C', str(PROJECT_ROOT), 'checkout', '-b', PROJECT_GIT_REF,
            '--track', f'origin/{PROJECT_GIT_REF}',
        ])

required_project_files = [
    PROJECT_ROOT / 'src/silent_signal/cli/select_subset.py',
    PROJECT_ROOT / 'src/silent_signal/cli/extract_pose.py',
    PROJECT_ROOT / 'configs/pose/rtmpose.yaml',
]
missing = [str(path) for path in required_project_files if not path.is_file()]
if missing:
    raise RuntimeError('Nhánh này chưa có pipeline top-200: ' + ', '.join(missing))
PROJECT_COMMIT = git_output(PROJECT_ROOT, 'rev-parse', 'HEAD')
print('Project commit:', PROJECT_COMMIT)


## GPU và môi trường RTMPose độc lập

Notebook tạo Python 3.11 riêng trong `/content/pose-env` và cài đúng bộ NumPy/PyTorch/MMCV/MMPose đã kiểm chứng. Kernel Colab không import `numpy`, `pandas` hoặc `pyarrow`, nhờ đó tránh lỗi ABI `numpy.dtype size changed`. Cell sẽ bỏ qua phần cài nặng nếu marker môi trường hợp lệ đã tồn tại trong runtime hiện tại.


In [ ]:
import os
import shutil
import sys

run(['nvidia-smi'])
disk = shutil.disk_usage('/content')
print(f'/content còn trống: {disk.free / 1024**3:.2f} GiB')
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
run(['uv', 'python', 'install', '3.11'])
if not (POSE_ENV_ROOT / 'bin/python').is_file():
    run(['uv', 'venv', str(POSE_ENV_ROOT), '--python', '3.11', '--seed'])

POSE_PY = str(POSE_ENV_ROOT / 'bin/python')
POSE_CLI = str(POSE_ENV_ROOT / 'bin/ss-extract-pose')
PREPARE_CLI = str(POSE_ENV_ROOT / 'bin/ss-prepare')
SUBSET_CLI = str(POSE_ENV_ROOT / 'bin/ss-select-asl-subset')
ENV_MARKER = POSE_ENV_ROOT / '.silent-signal-rtmpose-v1.ready'

if not MMPOSE_ROOT.exists():
    run([
        'git', 'clone', '--depth', '1', '--branch', 'v1.3.2',
        'https://github.com/open-mmlab/mmpose.git', str(MMPOSE_ROOT),
    ])
elif not (MMPOSE_ROOT / '.git').is_dir():
    raise RuntimeError('MMPOSE_ROOT tồn tại nhưng không phải source MMPose.')

if not ENV_MARKER.is_file():
    run([POSE_PY, '-m', 'pip', 'install',
         'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1'])
    run([POSE_PY, '-m', 'pip', 'install', 'numpy==1.26.4'])
    run([POSE_PY, '-m', 'pip', 'install', '--no-build-isolation', 'chumpy==0.70'])
    run([
        POSE_PY, '-m', 'pip', 'install', 'torch==2.1.0', 'torchvision==0.16.0',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
        '--extra-index-url', 'https://pypi.org/simple',
    ])
    run([POSE_PY, '-m', 'pip', 'install', 'mmengine==0.10.7'])
    run([
        POSE_PY, '-m', 'pip', 'install', 'mmcv==2.1.0',
        '-f', 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html',
    ])
    run([POSE_PY, '-m', 'pip', 'install', 'mmdet==3.2.0'])
    run([POSE_PY, '-m', 'pip', 'install', '-e', str(MMPOSE_ROOT)])

run([POSE_PY, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)])
environment_check = r'''
import cv2
import mmcv
import mmengine
import mmdet
import mmpose
import numpy as np
import pyarrow
import torch
from mmcv.ops import nms
assert torch.cuda.is_available(), 'CUDA không khả dụng trong pose-env.'
assert torch.ones(1, device='cuda:0').is_cuda
print('GPU:', torch.cuda.get_device_name(0))
print('NumPy:', np.__version__, 'PyArrow:', pyarrow.__version__)
print('MMCV:', mmcv.__version__, 'MMPose:', mmpose.__version__)
print('mmcv.ops: OK')
'''
run([POSE_PY, '-c', environment_check])
ENV_MARKER.write_text(PROJECT_COMMIT + '\n', encoding='utf-8')

PROCESS_ENV = os.environ.copy()
PROCESS_ENV['ASL_CITIZEN_ROOT'] = str(DATASET_ROOT)
PROCESS_ENV['MMPOSE_ROOT'] = str(MMPOSE_ROOT)
PROCESS_ENV['RTMPOSE_L_WHOLEBODY_CHECKPOINT'] = str(POSE_CHECKPOINT)
PROCESS_ENV['RTMDET_M_PERSON_CHECKPOINT'] = str(DET_CHECKPOINT)
PROCESS_ENV['MPLBACKEND'] = 'Agg'


## Lấy ASL Citizen và tự tạo manifest

Nếu `DATASET_ROOT` đã có đủ `videos/` và ba CSV split, notebook dùng lại ngay. Nếu thiếu và `STREAM_EXTRACT_DATASET=True`, notebook đọc ZIP chính thức bằng HTTP Range và giải nén thẳng vào `/content`, không giữ file ZIP 42,8 GiB. Dataset sau giải nén cần khoảng 46,2 GiB cộng 2 GiB dự phòng. Việc tải chỉ bắt đầu khi bạn bật `ACCEPT_ASL_CITIZEN_LICENSE`.

Điều khoản: [Microsoft Research Data License Agreement](https://www.microsoft.com/en-us/research/project/asl-citizen/dataset-license/). Dataset trong `/content` mất khi runtime reset; pose cache và metadata trên Drive vẫn được giữ.


In [ ]:
required_dataset_paths = [
    DATASET_ROOT / 'videos',
    DATASET_ROOT / 'splits/train.csv',
    DATASET_ROOT / 'splits/val.csv',
    DATASET_ROOT / 'splits/test.csv',
]
missing_dataset_paths = [str(path) for path in required_dataset_paths if not path.exists()]
if missing_dataset_paths:
    if not STREAM_EXTRACT_DATASET:
        raise FileNotFoundError(
            'Dataset chưa có và STREAM_EXTRACT_DATASET=False: '
            + ', '.join(missing_dataset_paths)
        )
    if not ACCEPT_ASL_CITIZEN_LICENSE:
        raise RuntimeError(
            'Đọc điều khoản Microsoft rồi bật ACCEPT_ASL_CITIZEN_LICENSE=True.'
        )
    extraction_code = (
        'import os; from pathlib import Path; '
        'from silent_signal.data.asl_download import extract_remote_archive; '
        "extract_remote_archive(Path(os.environ['ASL_CITIZEN_ROOT']))"
    )
    run([POSE_PY, '-c', extraction_code], env=PROCESS_ENV)

missing_dataset_paths = [str(path) for path in required_dataset_paths if not path.exists()]
if missing_dataset_paths:
    raise FileNotFoundError('ASL Citizen chưa đầy đủ: ' + ', '.join(missing_dataset_paths))
print('ASL Citizen sẵn sàng:', DATASET_ROOT)


In [ ]:
import json
import yaml

SOURCE_DATASET_CONFIG = PROJECT_ROOT / 'configs/dataset/asl_citizen.yaml'
RUNTIME_DATASET_CONFIG = PREPARATION_ROOT / 'configs/asl_citizen.top200.colab.yaml'
RUNTIME_DATASET_CONFIG.parent.mkdir(parents=True, exist_ok=True)
dataset_config = yaml.safe_load(SOURCE_DATASET_CONFIG.read_text(encoding='utf-8'))
dataset_config['dataset']['root'] = str(DATASET_ROOT)
dataset_config['outputs'] = {
    'manifest_csv': str(SOURCE_MANIFEST),
    'manifest_parquet': str(PREPARATION_ROOT / 'manifests/asl_citizen.parquet'),
    'labels': str(PREPARATION_ROOT / 'labels/asl_citizen_labels.json'),
    'split': str(PREPARATION_ROOT / 'splits/asl_citizen_official.json'),
    'report': str(PREPARATION_ROOT / 'reports/top200_metadata_validation.json'),
    'invalid_records': str(PREPARATION_ROOT / 'reports/top200_metadata_invalid.csv'),
}
RUNTIME_DATASET_CONFIG.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True), encoding='utf-8'
)
if PREPARE_MANIFEST:
    run([
        PREPARE_CLI, 'all', '--config', str(RUNTIME_DATASET_CONFIG),
        '--root', str(DATASET_ROOT), '--level', 'metadata',
    ], cwd=PROJECT_ROOT, env=PROCESS_ENV)
elif not SOURCE_MANIFEST.is_file():
    raise FileNotFoundError('PREPARE_MANIFEST=False nhưng chưa có source manifest trên Drive.')
print('Source manifest:', SOURCE_MANIFEST)


## Tải model và tạo pinned config

Checkpoint được giữ trong Drive. Notebook tính SHA-256 thực tế rồi tạo config có hash cố định; extraction không chấp nhận âm thầm đổi model.


In [ ]:
POSE_URL = (
    'https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/'
    'rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-'
    'eaeb96c8_20230125.pth'
)
DET_URL = (
    'https://download.openmmlab.com/mmpose/v1/projects/rtmpose/'
    'rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth'
)

def download_checkpoint(url, destination):
    if destination.is_file():
        print('Đã tồn tại:', destination)
        return
    partial = Path(str(destination) + '.part')
    run(['wget', '-c', url, '-O', str(partial)])
    if not partial.is_file() or partial.stat().st_size == 0:
        raise RuntimeError(f'Download checkpoint rỗng: {partial}')
    partial.replace(destination)

if DOWNLOAD_MODELS:
    download_checkpoint(POSE_URL, POSE_CHECKPOINT)
    download_checkpoint(DET_URL, DET_CHECKPOINT)
required_models = [POSE_CHECKPOINT, DET_CHECKPOINT]
missing_models = [str(path) for path in required_models if not path.is_file()]
if missing_models:
    raise FileNotFoundError('Thiếu checkpoint: ' + ', '.join(missing_models))
print('Model sẵn sàng:', MODEL_ROOT)


In [ ]:
SOURCE_POSE_CONFIG = PROJECT_ROOT / 'configs/pose/rtmpose.yaml'
PROVENANCE_ROOT = PINNED_CONFIG.parent
PROVENANCE_ROOT.mkdir(parents=True, exist_ok=True)
UNPINNED_LOCK = PROVENANCE_ROOT / 'unverified-hashes.lock.json'
PINNED_LOCK = PROVENANCE_ROOT / 'rtmpose-colab-pinned.lock.json'
run([
    POSE_CLI, 'verify', '--config', str(SOURCE_POSE_CONFIG),
    '--write-lock', str(UNPINNED_LOCK),
], env=PROCESS_ENV)
unverified = json.loads(UNPINNED_LOCK.read_text(encoding='utf-8'))
pose_config = yaml.safe_load(SOURCE_POSE_CONFIG.read_text(encoding='utf-8'))
pose_config['extractor']['pose_model']['checkpoint_sha256'] = (
    unverified['pose_model']['checkpoint_sha256']
)
pose_config['extractor']['detector']['checkpoint_sha256'] = (
    unverified['detector']['checkpoint_sha256']
)
PINNED_CONFIG.write_text(
    yaml.safe_dump(pose_config, sort_keys=False, allow_unicode=True), encoding='utf-8'
)
run([
    POSE_CLI, 'verify', '--config', str(PINNED_CONFIG),
    '--write-lock', str(PINNED_LOCK),
], env=PROCESS_ENV)
print('Pinned config:', PINNED_CONFIG)


## Tải bảng tần suất ASL-LEX 2.0

File CSV nhỏ được lưu trên Drive để các lần chạy sau dùng lại. Source ASL Citizen và source ASL-LEX đều được băm SHA-256 trong báo cáo selection; dữ liệu nguồn không được đưa vào Git.


In [ ]:
if not ASL_LEX_CSV.is_file():
    partial = ASL_LEX_CSV.with_suffix('.csv.part')
    run(['wget', '-c', ASL_LEX_URL, '-O', str(partial)])
    if not partial.is_file() or partial.stat().st_size < 100_000:
        raise RuntimeError('ASL-LEX CSV tải về rỗng hoặc không đúng định dạng mong đợi.')
    partial.replace(ASL_LEX_CSV)
print('ASL-LEX CSV:', ASL_LEX_CSV, ASL_LEX_CSV.stat().st_size, 'bytes')


## Chọn đúng 200 lớp và ghi manifest lên Drive

Mọi clip thuộc 200 lớp được giữ lại ở đúng split chính thức. Chỉ `class_index` được ánh xạ lại liên tục từ 0 đến 199; `selection_report.json` giữ class index gốc, code ASL-LEX, điểm tần suất và số clip từng split.


In [ ]:
selection_command = [
    str(SUBSET_CLI),
    '--manifest', str(SOURCE_MANIFEST),
    '--asl-lex-csv', str(ASL_LEX_CSV),
    '--classes', str(CLASS_COUNT),
    '--output-root', str(SUBSETS_ROOT),
    '--project-commit', PROJECT_COMMIT,
]
if RUN_SELECTION:
    run(selection_command, env=PROCESS_ENV)
elif not SUBSET_MANIFEST.is_file() or not SELECTION_REPORT.is_file():
    raise FileNotFoundError('RUN_SELECTION=False nhưng chưa có subset trên Drive.')
else:
    print('Dùng lại subset đã có:', SUBSET_ROOT)


In [ ]:
import json

selection = json.loads(SELECTION_REPORT.read_text(encoding='utf-8'))
classes = selection['classes']
if selection['classes_selected'] != CLASS_COUNT:
    raise RuntimeError('Selection không tạo đúng số lớp yêu cầu.')
if [item['subset_class_index'] for item in classes] != list(range(CLASS_COUNT)):
    raise RuntimeError('Subset class index không liên tục từ 0.')
if selection['subset_manifest']['glosses'] != CLASS_COUNT:
    raise RuntimeError('Manifest subset không chứa đúng số lớp.')
if selection['criterion']['column'] != 'SignFrequency(M)':
    raise RuntimeError('Selection report dùng sai tiêu chí tần suất.')
frequency_scores = [item['sign_frequency_mean'] for item in classes]
if frequency_scores != sorted(frequency_scores, reverse=True):
    raise RuntimeError('Danh sách lớp không được xếp giảm dần theo ASL-LEX frequency.')
if [item['rank'] for item in classes] != list(range(1, CLASS_COUNT + 1)):
    raise RuntimeError('Rank top-200 không liên tục từ 1.')

print(json.dumps({
    'subset': selection['subset'],
    'classes': selection['classes_selected'],
    'unique_asl_lex_codes': selection['unique_asl_lex_codes_selected'],
    'shared_asl_lex_codes': selection['shared_asl_lex_codes'],
    'clips': selection['subset_manifest']['clips'],
    'splits': selection['subset_manifest']['splits'],
    'excluded_class_counts': selection['excluded_class_counts'],
    'source_hashes': {key: value['sha256'] for key, value in selection['sources'].items()},
}, ensure_ascii=False, indent=2))
print('\nTop 30:')
print(f"{'rank':>4}  {'gloss':<28} {'frequency':>9}  {'clips':>6}")
for item in classes[:30]:
    print(
        f"{item['rank']:>4}  {item['gloss_name'][:28]:<28} "
        f"{item['sign_frequency_mean']:>9.3f}  {item['clip_count']:>6}"
    )


## Pilot 20 video

Pilot dùng 20 video train đầu tiên trong subset. Cache được ghi thẳng vào thư mục raw chính; full extraction sau đó kiểm tra provenance và resume các cache hợp lệ. Nếu cache cũ sai model/source, CLI dừng và yêu cầu kiểm tra hoặc bật `OVERWRITE`.


In [ ]:
def run_pose(command, report_path, label):
    print('+', ' '.join(str(part) for part in command))
    completed = subprocess.run(command, env=PROCESS_ENV)
    if not report_path.is_file():
        raise RuntimeError(f'{label} không tạo report: {report_path}')
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({
        key: value for key, value in report.items() if key != 'failures'
    }, ensure_ascii=False, indent=2))
    if completed.returncode != 0 or report['failed']:
        print(json.dumps(report['failures'][:20], ensure_ascii=False, indent=2))
        raise RuntimeError(f'{label} có video lỗi; xem report đầy đủ trên Drive.')
    return report

PILOT_REPORT = POSE_ROOT / f'pilot_train_{PILOT_LIMIT}.json'
pilot_command = [
    str(POSE_CLI), 'extract',
    '--config', str(PINNED_CONFIG),
    '--manifest', str(SUBSET_MANIFEST),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(POSE_OUTPUT_ROOT),
    '--report', str(PILOT_REPORT),
    '--split', 'train', '--limit', str(PILOT_LIMIT),
    '--device', 'cuda:0', '--progress-every', '1',
]
if OVERWRITE:
    pilot_command.append('--overwrite')
if RUN_PILOT:
    run_pose(pilot_command, PILOT_REPORT, 'Pilot')
else:
    print('RUN_PILOT=False — bỏ qua pilot.')


## Full extraction toàn bộ train/validation/test

Chỉ bật `RUN_FULL_EXTRACTION=True` sau khi pilot thành công và đã kiểm tra số clip trong báo cáo. Mặc định `NUM_SHARDS=1`. Nếu chạy nhiều Colab worker, đặt cùng `NUM_SHARDS` và một `SHARD_INDEX` khác nhau cho từng worker; mỗi sample được gán tất định vào đúng một shard. Không dùng `--split`, vì cần extract toàn bộ ba split của 200 lớp.


In [ ]:
FULL_REPORT = POSE_ROOT / f'full_shard_{SHARD_INDEX:03d}_of_{NUM_SHARDS:03d}.json'
full_command = [
    str(POSE_CLI), 'extract',
    '--config', str(PINNED_CONFIG),
    '--manifest', str(SUBSET_MANIFEST),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(POSE_OUTPUT_ROOT),
    '--report', str(FULL_REPORT),
    '--device', 'cuda:0',
    '--num-shards', str(NUM_SHARDS),
    '--shard-index', str(SHARD_INDEX),
    '--continue-on-error', '--progress-every', '25',
]
if OVERWRITE:
    full_command.append('--overwrite')
if RUN_FULL_EXTRACTION:
    run_pose(full_command, FULL_REPORT, 'Full extraction')
else:
    print('RUN_FULL_EXTRACTION=False — chưa chạy toàn bộ dữ liệu.')
    print('Sau khi pilot ổn, bật cờ ở cell cấu hình rồi chạy lại cell này.')


## Kết quả trên Drive

- `subsets/asl_citizen_asllex_top200/manifest.csv|parquet`: toàn bộ clip của 200 lớp.
- `labels.json`: ánh xạ 200 nhãn mới.
- `selection_report.json`: tiêu chí, hash nguồn, bảng xếp hạng và số clip/split.
- `pose/rtmpose_l_coco_wholebody_384x288/raw/`: cache keypoint nguyên bản, có thể resume.
- `pilot_*.json` và `full_shard_*.json`: báo cáo tiến độ/lỗi extraction.

Không commit dataset, ASL-LEX CSV, manifest sinh ra hoặc pose cache vào Git. Chỉ commit mã nguồn, notebook không output và tài liệu.
